# Entrenamiento YOLOv8n — Detector de Tablas en Extractos

**Antes de correr:** Ir a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

**Flujo completo:**
1. Instalar dependencias
2. Subir dataset desde Drive
3. Verificar estructura
4. Entrenar
5. Evaluar métricas
6. Exportar ONNX
7. Descargar modelo

## Celda 1 — Instalar dependencias y verificar GPU

In [ ]:
!pip install ultralytics -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('ADVERTENCIA: No hay GPU. Ir a Entorno de ejecucion → Cambiar tipo → GPU T4')

## Celda 2 — Subir dataset desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Ajustar esta ruta segun donde subiste dataset.zip en tu Drive
ZIP_PATH = '/content/drive/MyDrive/dataset.zip'

if not os.path.exists(ZIP_PATH):
    print(f'ERROR: No se encontro {ZIP_PATH}')
    print('Sube dataset.zip a Google Drive y ajusta ZIP_PATH arriba.')
else:
    print(f'Archivo encontrado: {ZIP_PATH}')
    !cp '{ZIP_PATH}' /content/dataset.zip
    !unzip -q /content/dataset.zip -d /content/
    print('Dataset descomprimido correctamente.')

## Celda 3 — Verificar estructura del dataset

In [ ]:
import os
from pathlib import Path

# Buscar dataset.yaml automaticamente
yaml_candidates = list(Path('/content').rglob('dataset.yaml'))
if not yaml_candidates:
    print('ERROR: No se encontro dataset.yaml')
    print('Verifica que el zip tenga la estructura correcta.')
else:
    YAML_PATH = str(yaml_candidates[0])
    DATASET_DIR = str(yaml_candidates[0].parent)
    print(f'dataset.yaml encontrado en: {YAML_PATH}')

# Contar imagenes por split
for split in ['train', 'val', 'test']:
    img_dir = Path(DATASET_DIR) / 'images' / split
    lbl_dir = Path(DATASET_DIR) / 'labels' / split
    if img_dir.exists():
        n_imgs = len(list(img_dir.iterdir()))
        n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        n_con_tabla = sum(1 for t in lbl_dir.glob('*.txt') if t.read_text().strip())
        print(f'  {split:6s}: {n_imgs} imgs | {n_lbls} labels | {n_con_tabla} con tabla')
    else:
        print(f'  {split}: carpeta no encontrada')

print()
print('Contenido de dataset.yaml:')
print(open(YAML_PATH).read())

## Celda 4 — Entrenar YOLOv8n (~10 min con GPU T4)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # descarga pesos COCO preentrenados

results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,        # GPU
    patience=15,     # early stopping
    name='tabla_detector',
    project='/content/runs',
    exist_ok=True,
    verbose=True
)

print(f'Entrenamiento finalizado.')
print(f'Mejor modelo: /content/runs/tabla_detector/weights/best.pt')

## Celda 5 — Evaluar métricas sobre test set

In [ ]:
from ultralytics import YOLO
import numpy as np

model = YOLO('/content/runs/tabla_detector/weights/best.pt')

# Metricas sobre el split de TEST (el que incluye fotos de celular)
metrics = model.val(
    data=YAML_PATH,
    split='test',
    verbose=False
)

print('=' * 50)
print('METRICAS SOBRE TEST SET')
print('=' * 50)
print(f'mAP@0.5:       {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95:  {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'Precision:     {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
print(f'Recall:        {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')
print('=' * 50)

## Celda 6 — IoU por imagen en test set

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import numpy as np
import cv2

model = YOLO('/content/runs/tabla_detector/weights/best.pt')

def xywhn_to_xyxy(cx, cy, w, h, W, H):
    """Convierte formato YOLO normalizado a coordenadas absolutas."""
    x1 = int((cx - w/2) * W)
    y1 = int((cy - h/2) * H)
    x2 = int((cx + w/2) * W)
    y2 = int((cy + h/2) * H)
    return x1, y1, x2, y2

def calc_iou(boxA, boxB):
    """IoU entre dos cajas [x1,y1,x2,y2]."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

test_imgs = sorted(Path(DATASET_DIR, 'images', 'test').iterdir())
test_lbls = Path(DATASET_DIR, 'labels', 'test')

ious = []
fallos = []

for img_path in test_imgs:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    H, W = img.shape[:2]

    lbl_path = test_lbls / (img_path.stem + '.txt')
    gt_lines = lbl_path.read_text().strip().splitlines() if lbl_path.exists() else []
    gt_boxes = []
    for line in gt_lines:
        parts = line.split()
        if len(parts) == 5:
            _, cx, cy, w, h = map(float, parts)
            gt_boxes.append(xywhn_to_xyxy(cx, cy, w, h, W, H))

    # Prediccion del modelo
    preds = model.predict(str(img_path), conf=0.4, verbose=False)
    pred_boxes = []
    for box in preds[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        pred_boxes.append((x1, y1, x2, y2))

    # Si no hay GT ni prediccion -> imagen negativa correcta
    if not gt_boxes and not pred_boxes:
        continue

    # Si hay GT pero no prediccion -> fallo
    if gt_boxes and not pred_boxes:
        fallos.append((img_path.name, 'no_detecto', 0.0))
        ious.append(0.0)
        continue

    # Si hay prediccion pero no GT -> falso positivo
    if not gt_boxes and pred_boxes:
        fallos.append((img_path.name, 'falso_positivo', 0.0))
        continue

    # Calcular IoU del mejor par GT-prediccion
    best_iou = max(calc_iou(gt, pred) for gt in gt_boxes for pred in pred_boxes)
    ious.append(best_iou)
    if best_iou < 0.5:
        fallos.append((img_path.name, 'iou_bajo', best_iou))

print('=' * 50)
print('IoU POR IMAGEN (test set)')
print('=' * 50)
if ious:
    print(f'IoU promedio: {np.mean(ious):.4f}')
    print(f'IoU minimo:   {np.min(ious):.4f}')
    print(f'IoU maximo:   {np.max(ious):.4f}')
    print(f'IoU >= 0.5:   {sum(1 for i in ious if i >= 0.5)}/{len(ious)} imagenes')
else:
    print('No se calcularon IoUs (revisar labels del test set)')

if fallos:
    print(f'\nFallos detectados ({len(fallos)}):')
    for nombre, tipo, iou in fallos:
        print(f'  {nombre:40s} | {tipo:20s} | IoU={iou:.3f}')
print('=' * 50)

## Celda 7 — Visualizar predicciones sobre fotos de celular

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
from IPython.display import Image as IPImage, display
import tempfile

model = YOLO('/content/runs/tabla_detector/weights/best.pt')

# Mostrar predicciones sobre las fotos reales de celular
test_dir = Path(DATASET_DIR, 'images', 'test')
celular_prefixes = ('foto', 'Foto', 'impresion', 'Impresion', 'whatsapp', 'WhatsApp')
celular_imgs = [p for p in test_dir.iterdir() if p.name.startswith(celular_prefixes)]

print(f'Fotos de celular en test set: {len(celular_imgs)}')

for img_path in celular_imgs[:6]:  # Mostrar las primeras 6
    results = model.predict(str(img_path), conf=0.4, verbose=False)
    annotated = results[0].plot()

    # Redimensionar para visualizar bien en Colab
    h, w = annotated.shape[:2]
    scale = 800 / max(h, w)
    vis = cv2.resize(annotated, (int(w*scale), int(h*scale)))

    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
        cv2.imwrite(f.name, vis)
        n_det = len(results[0].boxes)
        print(f'\n{img_path.name} — {n_det} tabla(s) detectada(s)')
        display(IPImage(f.name))

## Celda 8 — Comparativa modelo vs detector morfológico anterior

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import time
import numpy as np

model = YOLO('/content/runs/tabla_detector/weights/best.pt')
test_dir = Path(DATASET_DIR, 'images', 'test')
test_imgs = list(test_dir.iterdir())

# Medir latencia del modelo YOLOv8n
tiempos = []
for img_path in test_imgs:
    t0 = time.time()
    model.predict(str(img_path), conf=0.4, verbose=False)
    tiempos.append((time.time() - t0) * 1000)

print('=' * 50)
print('COMPARATIVA DE RENDIMIENTO')
print('=' * 50)
print(f'YOLOv8n (GPU Colab):')
print(f'  Latencia promedio: {np.mean(tiempos):.1f} ms/imagen')
print(f'  Latencia maxima:   {np.max(tiempos):.1f} ms/imagen')
print()
print('Detector morfologico OpenCV (referencia):')
print('  Latencia:   ~5-20 ms/imagen  (solo en PDFs limpios)')
print('  Limitacion: falla en fotos de celular con perspectiva')
print()
print('YOLOv8n en CPU local (estimado para Docker):')
print('  Latencia:   <100 ms/imagen')
print('  Ventaja:    funciona en fotos de celular y PDFs')
print('=' * 50)

## Celda 9 — Exportar a ONNX (opset 11, compatible con OpenCV 4.13)

In [ ]:
from ultralytics import YOLO
import os

model = YOLO('/content/runs/tabla_detector/weights/best.pt')

model.export(
    format='onnx',
    opset=11,        # compatible con OpenCV 4.13 cv::dnn
    simplify=True,   # reduce operaciones redundantes
    imgsz=640
)

onnx_path = '/content/runs/tabla_detector/weights/best.onnx'
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f'ONNX exportado: {onnx_path}')
print(f'Tamaño: {size_mb:.1f} MB')

## Celda 10 — Descargar modelo ONNX

In [ ]:
from google.colab import files

onnx_path = '/content/runs/tabla_detector/weights/best.onnx'
files.download(onnx_path)

print('Descargando best.onnx...')
print()
print('Cuando termine la descarga, ejecutar en tu maquina local:')
print('  mkdir -p models')
print('  cp ~/Downloads/best.onnx models/tabla_detector.onnx')

## Celda 11 — Guardar resultados en Drive (opcional)

In [ ]:
import shutil

# Guardar modelo y resultados en Drive para no perderlos
dest = '/content/drive/MyDrive/tabla_detector_results'
shutil.copytree('/content/runs/tabla_detector', dest, dirs_exist_ok=True)
print(f'Resultados guardados en Drive: {dest}')
print('Incluye: best.pt, best.onnx, graficas de entrenamiento, metricas')